# Phase 3: Exploratory Data Analysis (EDA)
**Project:** Customer Churn Prediction & Behavioral Segmentation  
**Objective:** Uncover behavioral patterns, cohort churn differentials, bivariate relationships, and statistical correlations.


In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = (10, 5)

DATA_PATH = os.path.join('..', 'data', 'processed', 'telco_cleaned.csv')
if not os.path.exists(DATA_PATH):
    DATA_PATH = os.path.join('data', 'processed', 'telco_cleaned.csv')

df = pd.read_csv(DATA_PATH)
print(f"Loaded dataset: {df.shape[0]} rows x {df.shape[1]} columns")


## 1. Churn Distribution & Class Balance

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
counts = df['Churn'].value_counts()
pcts = df['Churn'].value_counts(normalize=True) * 100

axes[0].bar(counts.index, counts.values, color=['#1f77b4', '#d62728'], width=0.4)
axes[0].set_title('Customer Count by Churn Status', fontweight='bold')
axes[0].set_ylabel('Customer Count')

axes[1].pie(pcts, labels=pcts.index, autopct='%1.2f%%', colors=['#1f77b4', '#d62728'], startangle=90, explode=[0, 0.08])
axes[1].set_title('Churn Rate Proportion', fontweight='bold')
plt.show()


## 2. Key Bivariate Drivers: Contract, Payment Method & Fiber Optic

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for i, col in enumerate(['Contract', 'PaymentMethod', 'InternetService']):
    ct = pd.crosstab(df[col], df['Churn'], normalize='index') * 100
    ct[['No', 'Yes']].plot(kind='bar', ax=axes[i], color=['#1f77b4', '#d62728'], edgecolor='black', alpha=0.85)
    axes[i].set_title(f'Churn by {col} (%)', fontweight='bold')
    axes[i].set_ylabel('Percentage (%)')
    axes[i].tick_params(axis='x', rotation=25)

plt.tight_layout()
plt.show()


## 3. Tenure & Monthly Charges vs Churn

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.boxplot(x='Churn', y='tenure', hue='Churn', data=df, ax=axes[0], palette=['#1f77b4', '#d62728'], legend=False)
axes[0].set_title('Tenure Distribution: Churn vs Retained', fontweight='bold')

sns.kdeplot(data=df, x='MonthlyCharges', hue='Churn', common_norm=False, fill=True,
            palette={'No': '#1f77b4', 'Yes': '#d62728'}, ax=axes[1], alpha=0.4)
axes[1].set_title('Monthly Charges Density', fontweight='bold')
plt.show()


## 4. Correlation Matrix

In [ ]:
df_corr = df.copy()
df_corr['Churn_Binary'] = (df_corr['Churn'] == 'Yes').astype(int)
df_corr['SeniorCitizen'] = df_corr['SeniorCitizen'].astype(int)
df_corr['Partner_Binary'] = (df_corr['Partner'] == 'Yes').astype(int)
df_corr['Dependents_Binary'] = (df_corr['Dependents'] == 'Yes').astype(int)
df_corr['PaperlessBilling_Binary'] = (df_corr['PaperlessBilling'] == 'Yes').astype(int)

num_cols = ['tenure', 'MonthlyCharges', 'TotalCharges', 'SeniorCitizen',
            'Partner_Binary', 'Dependents_Binary', 'PaperlessBilling_Binary', 'Churn_Binary']

plt.figure(figsize=(10, 7))
sns.heatmap(df_corr[num_cols].corr(), annot=True, fmt='.2f', cmap='coolwarm', vmin=-1, vmax=1)
plt.title('Correlation Matrix of Numerical & Demographic Features', fontweight='bold')
plt.show()
